# FirstTouch - Inbound Lead Triage Agent Team

_by Mark Takla_

A believable team of AI agents that takes a raw inbound lead, enriches it, **debates** whether to pursue it, resolves the debate with an explicit rule, and (if pursued) drafts outreach and writes a clean CRM record - with a full readable trace of who said what and why.

**Stack:** Python + LangChain, free Llama 3.x on **Groq** (`langchain-groq`), Pydantic for typed contracts.

**Non-negotiable design principle - debate first:** before any consequential action (drafting/"sending" outreach), two agents with genuinely different objectives put forward competing recommendations, and a deterministic Arbiter resolves the disagreement and records the dissent.

---

## The team

| Agent | Responsibility |
|---|---|
| **Lead Intelligence** | Enriches the raw lead (mock Apollo/Clay/Hunter), scores it against the ICP, proposes a disposition. |
| **Growth Advocate** | Argues *for* pursuing - the optimistic, revenue-seeking case. |
| **Risk Skeptic** (devil's advocate) | Argues *against* - hunts red flags, fit gaps, and the cost of a clumsy first-touch. |
| **Arbiter** (deterministic) | Applies the resolution rule: critic veto -> confidence-weighted vote -> bounded rebuttal -> escalation. |
| **SDR / Outreach** | If the team decides to pursue, drafts a personalised email + LinkedIn note grounded in the enriched data. |
| **CRM Hygiene** | Writes a clean, structured CRM record (mock HubSpot), sets the deal stage, flags anything needing a human. |
| **Orchestrator** | Owns shared state, runs the sequence, triggers the debate, applies the resolution rule, routes handoffs. |

**Flow:** intake -> enrich -> debate the disposition -> resolve & record -> handoff to outreach (if pursue) -> handoff to CRM -> output artifacts + a readable trace.

> Business reasoning, ICP rationale, and interview prep live in `SIRCLES_CONTEXT.md`. The 1-page design note lives in `DESIGN_NOTE.md`.


## 1. Setup & config

Install dependencies (run once), load the Groq API key from `.env`, and build the shared LLM client.

- The key is read from the `GROQ_API_KEY` environment variable - **never hard-code it in the notebook**.
- Model defaults to `llama-3.3-70b-versatile` (free on Groq). Override with `GROQ_MODEL`.
- We set `temperature=0.2` so the deterministic logic dominates and runs are reproducible enough to discuss.

In [1]:
# Run once to install dependencies (uncomment if needed):
# %pip install -r requirements.txt

import os
from dotenv import load_dotenv

import sircles_core as core
from sircles_core import (
    RawLead, Enrichment, ICPScore, Position, Resolution, OutreachArtifacts, CRMRecord, RunState,
    fetch_new_leads, enrich_contact, enrich_company, upsert_crm_record,
    score_icp, resolve, is_contested, build_llm, build_agents,
    run_pipeline, run_pipeline_events, render_trace, lead_brief,
)

load_dotenv()
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError(
        "GROQ_API_KEY is not set. Copy .env.example to .env and add your free key "
        "from https://console.groq.com/keys"
    )

# All implementation lives in sircles_core.py (single source of truth shared with the UI, app.py).
llm = build_llm(GROQ_MODEL)
agents = build_agents(llm)
print(f"LLM ready: {GROQ_MODEL}")


LLM ready: llama-3.3-70b-versatile


## 2. Schemas & shared state

Typed contracts are how the agents hand off to each other cleanly. Every boundary in the system is a Pydantic model, so a malformed agent output fails loudly instead of silently corrupting downstream steps.

- **`RawLead`** - what comes in from the scanner.
- **`Enrichment`** (`ContactEnrichment` + `CompanyEnrichment`) - what the enrichment tools return.
- **`ICPScore`** - the deterministic fit score, band, and disqualifiers.
- **`Position`** - one agent's debate stance: recommendation + confidence + evidence + risks.
- **`Resolution`** - the Arbiter's verdict, including recorded dissent and the escalation flag.
- **`OutreachArtifacts`** and **`CRMRecord`** - the final deliverables.
- **`RunState`** - the shared blackboard the Orchestrator threads through every step, plus the trace.

In [2]:
# Typed contracts are defined in sircles_core.py (imported above). Quick look at two of them:
print("Position fields:", list(Position.model_fields))
print("RunState fields:", list(RunState.model_fields))


Position fields: ['agent', 'recommendation', 'confidence', 'rationale', 'key_evidence', 'risks']
RunState fields: ['raw_lead', 'enrichment', 'icp', 'positions', 'rebuttals', 'resolution', 'outreach', 'crm_record', 'trace']


## 3. Mock tools & fixtures

No live APIs. We stub the four tools the spec asks for, backed by in-notebook fixtures:

- **`fetch_new_leads()`** - the lead scanner (returns raw inbound leads).
- **`enrich_contact()`** / **`enrich_company()`** - mock Apollo/Clay/Hunter enrichment.
- **`upsert_crm_record()`** - mock HubSpot write (appends to an in-memory store).

### The four sample leads (chosen to exercise the decision boundary)

1. **`L-001` clear pursue** - VP Marketing at a Cairo-based Series B B2B SaaS, hiring a growth marketer, no in-house agency.
2. **`L-002` clear skip** - solo "freelance consultant" on a Gmail address, no budget signals.
3. **`L-003` ambiguous (forces debate)** - Head of Growth at *BrightWave Digital*: strong role/size/industry/location fit, but the company **is itself a marketing agency** (a competitor). Great on paper, conflict in reality.
4. **`L-004` nurture** - early-stage Egyptian founder, good fit but pre-funding and too small *right now*.

In [3]:
# Mock tools + fixtures live in sircles_core.py. The four sample leads:
for _l in fetch_new_leads():
    print(_l.lead_id, "-", _l.name, "@", _l.company)


L-001 - Mariam Hassan @ NileBooks
L-002 - Tarek M. @ Self-employed
L-003 - Omar Farouk @ BrightWave Digital
L-004 - Yasmine Adel @ Zeitoun


## 4. ICP scoring (deterministic)

The fit score is **deterministic, not LLM-generated** - so it is auditable, testable, and consistent. The LLM agents *reason about* this score in the debate, but they cannot silently change it.

### Rubric (0-100)

| Dimension | Max | Why it matters for a marketing agency |
|---|---|---|
| Decision-maker seniority | 20 | A retainer needs an economic buyer (CMO/VP/Founder), not an IC. |
| Company size / budget fit | 20 | Retainers need budget; too small can't pay, too big has in-house teams. |
| Industry fit | 20 | We win in target verticals (B2B SaaS, e-commerce, fintech, consumer apps...). |
| **Location fit (Egypt-weighted)** | 15 | Sircles operates from **Egypt**: same timezone, language, payment rails, local market knowledge. Egypt = full marks, MENA = partial, far-geo = low (not disqualifying). |
| Growth / intent signals | 15 | Funding, hiring marketers, traffic growth = ready to spend now. |
| Marketing maturity gap | 10 | No/small in-house team = they actually need us. |

**Bands:** `>= 70 -> pursue`, `40-69 -> nurture`, `< 40 -> skip`.

**Budget-readiness rule:** an explicit "no budget yet" signal caps an otherwise strong-fit lead at **nurture** - a great future client we can't close *today*. (This is why the pre-seed founder `L-004` lands in nurture despite a high raw score.)

### Hard disqualifiers (override the score)

- **`competing_agency`** - the lead is itself a marketing agency. Not auto-skip: routed to a human (could be a partnership, could be a competitor fishing).
- **`student_jobseeker`** - title signals a job-seeker/student, not a buyer.
- **`no_budget_micro`** - free-email micro-entity with no budget signals.

### Confidence vs. score

Score answers *"is this a good fit?"*. **Confidence** answers *"how sure are we?"* and is driven by **data completeness** - missing email/LinkedIn/firmographics lower it. Data quality therefore shapes confidence (and escalation), not the fit score itself.

In [4]:
# Deterministic ICP scoring lives in sircles_core.py. Demo on the clear-pursue lead L-001:
_lead = {l.lead_id: l for l in fetch_new_leads()}["L-001"]
_icp = score_icp(Enrichment(contact=enrich_contact(_lead), company=enrich_company(_lead)))
print("L-001 ICP:", _icp.total, "band:", _icp.band)
print("breakdown:", _icp.breakdown)
print("disqualifiers:", _icp.disqualifiers, "| data_completeness:", _icp.data_completeness)


L-001 ICP: 100 band: pursue
breakdown: {'seniority': 20, 'size_budget': 20, 'industry': 20, 'location': 15, 'signals': 15, 'maturity_gap': 10}
disqualifiers: [] | data_completeness: 1.0


## 5. Agents

Five LLM-backed roles, each a thin LangChain runnable with a **typed output contract** (`with_structured_output`). The two debate agents are deliberately given **opposed objectives and different evidence lenses** - this is what makes the debate real rather than one prompt agreeing with itself.

- **Lead Intelligence** - turns the enrichment + ICP score into a tight intelligence brief that grounds the debate.
- **Growth Advocate** - default lens is opportunity/revenue; argues to pursue.
- **Risk Skeptic** - default lens is cost-of-wrong/reputation; hunts red flags, argues to skip/nurture.
- **SDR / Outreach** - drafts the email + LinkedIn note, grounded only in enriched facts.
- **CRM Hygiene** - assembles the structured record (deterministic fields + an LLM-written summary note).

A `rebuttal` variant lets each debate agent respond to the other's position once, for the bounded back-and-forth.

In [5]:
# The agent team is built from the injected LLM in sircles_core.build_agents().
# It was already created above as `agents` (Lead Intelligence, Growth Advocate, Risk Skeptic,
# SDR/Outreach, CRM Hygiene). Each is a thin LangChain runnable with a typed output contract.
print("Agent methods:", [m for m in dir(agents) if not m.startswith("_")])


Agent methods: ['advocate', 'crm_hygiene', 'lead_intelligence', 'llm', 'outreach', 'rebuttal', 'skeptic']


## 6. Debate & resolution (the Arbiter)

The Arbiter is **deterministic on purpose** - the resolution rule is the single most consequential piece of logic, so it must be auditable and unit-testable, never a black box. It consumes the two positions (and the ICP disqualifiers) and returns a `Resolution` that always records *who won, who lost, and why*.

### Resolution rule (in priority order)

1. **Critic veto - competing agency** -> escalate to a human. A competitor could be a conflict *or* a partnership; that is a human call, never an autopilot skip.
2. **Hard disqualifier** (job-seeker / no-budget micro) -> `skip` / archive with high confidence.
3. **Low data confidence** (`data_completeness < 0.5`) -> escalate; we don't act on thin data.
4. **Consensus** (both agents agree) -> take it, with a small confidence bonus.
5. **Clear winner** (they disagree but confidences differ by >= `MARGIN`) -> confidence-weighted vote; the loser's position is recorded as dissent.
6. **Contested** (disagree with comparable confidence) -> one **bounded rebuttal** round, then:
   - **Polar split** (pursue vs skip) still unresolved -> **escalate** (a genuine high-stakes deadlock is a human call).
   - **Adjacent split** (pursue/nurture or nurture/skip) -> the **ICP band breaks the tie** - no human needed. A strong-fit lead is not escalated just because the Skeptic prefers a softer touch.

### Confidence -> action gating

| Disposition | Confidence | Action |
|---|---|---|
| pursue | >= 0.70 | **auto_send** (no human in the loop) |
| pursue | < 0.70 | draft_for_review (human approves before send) |
| nurture | any | nurture_sequence |
| skip | any | archive |
| *escalate flag* | any | human_review |

The threshold reflects the cost asymmetry: a bad autonomous first-touch is irreversible, so we only let the system "send" when it is genuinely confident.

In [6]:
# The deterministic Arbiter resolution rule lives in sircles_core.resolve(...).
# Key thresholds (see sircles_core.py for the full priority-ordered rule):
print("MARGIN:", core.MARGIN, "| CONF_THRESHOLD:", core.CONF_THRESHOLD, "| MIN_DATA:", core.MIN_DATA)


MARGIN: 0.15 | CONF_THRESHOLD: 0.7 | MIN_DATA: 0.5


## 7. Orchestrator & trace

The Orchestrator is the team lead. It owns the shared `RunState`, runs the sequence, **triggers the debate, runs at most one rebuttal when contested**, applies the resolution rule, and routes handoffs:

- `pursue` -> SDR/Outreach drafts artifacts -> CRM Hygiene.
- `nurture` / `skip` -> straight to CRM Hygiene (no first-touch).
- escalate -> human_review (no autonomous outreach) -> CRM Hygiene flags it.

Every step appends to a structured `trace`, and `render_trace` turns it into a readable account a non-author can follow - **who said what, with what confidence, who won, and why the loser lost.**

In [7]:
# The event-emitting orchestrator (run_pipeline_events) plus the run_pipeline / render_trace
# helpers live in sircles_core.py. The Streamlit UI (app.py) consumes run_pipeline_events to
# render each step live; the notebook uses the run_pipeline wrapper below.
print("Orchestrator ready.")


Orchestrator ready.


## 8. Runs

Two required traces:

- **Clear case (`L-001`)** - strong fit; the agents should converge on pursue and the system drafts outreach autonomously.
- **Ambiguous case (`L-003`)** - strong fit on paper but a competing agency; the debate must surface the conflict and the Arbiter must escalate rather than auto-send.

A summary table of all four leads follows, to show the full decision boundary.

In [8]:
from IPython.display import Markdown, display

LEADS = {l.lead_id: l for l in fetch_new_leads()}

# --- Clear case: L-001 (expect pursue + autonomous outreach) ---
state_clear = run_pipeline(LEADS["L-001"], agents)
display(Markdown(render_trace(state_clear)))


# Run trace - L-001: Mariam Hassan @ NileBooks
_Source: Website demo request. Inbound note: Filled in the 'book a demo' form, mentioned scaling paid acquisition._

## 1. Enrichment & ICP score
- Contact: VP of Marketing, seniority `exec`, Cairo, Egypt, email verified: True
- Company: B2B SaaS (fintech), 120 employees, Series B, agency: False
- Signals: Hiring a growth marketer, Recent Series B, Website traffic up 40% QoQ
- **ICP score: 100/100** (band `pursue`) | breakdown: {'seniority': 20, 'size_budget': 20, 'industry': 20, 'location': 15, 'signals': 15, 'maturity_gap': 10}
- Disqualifiers: none | data completeness: 1.0

## 2. Debate (round 1)
- **Growth Advocate** -> `pursue` @ confidence **0.90**
  - Rationale: NileBooks' strong growth signals, high ICP score, and interest in scaling paid acquisition make them a prime candidate for Sircles' services. With a recent Series B funding and a growth marketer on board, they have the potential to become a multi-month retainer. While there is a risk of high expectations for rapid growth, Sircles can mitigate this by setting clear goals and delivering results quickly.
  - Evidence: NileBooks' recent Series B funding; 40% quarter-over-quarter increase in website traffic; ICP score of 100; Interest in scaling paid acquisition
  - Risks: High expectations for rapid growth; Potential difficulty in delivering results quickly
- **Risk Skeptic** -> `nurture` @ confidence **0.80**
  - Rationale: NileBooks appears to be a strong fit for Sircles' services, but the risk of high expectations for rapid growth poses a challenge for delivering results quickly
  - Evidence: ICP score of 100; Strong growth signals, including 40% quarter-over-quarter increase in website traffic; Recent Series B funding; Interest in scaling paid acquisition
  - Risks: High expectations for rapid growth; Potential difficulty in delivering results quickly

## 2b. Rebuttal round
- **Growth Advocate** -> `pursue` @ confidence **0.90**
  - Rationale: While the Risk Skeptic raises a valid concern about high expectations for rapid growth, I believe the benefits of pursuing NileBooks outweigh the risks. The company's strong growth signals, recent Series B funding, and interest in scaling paid acquisition demonstrate a clear need for Sircles' services. Furthermore, the ICP score of 100 and the company's seniority and size suggest a high potential for long-term revenue growth. To mitigate the risk of high expectations, I recommend a thorough onboarding process and regular progress updates to ensure alignment with NileBooks' goals.
  - Evidence: ICP score of 100; Strong growth signals, including 40% quarter-over-quarter increase in website traffic; Recent Series B funding; Interest in scaling paid acquisition
  - Risks: High expectations for rapid growth; Potential difficulty in delivering results quickly
- **Risk Skeptic** -> `nurture` @ confidence **0.70**
  - Rationale: While NileBooks' strong growth signals and high ICP score are promising, the risk of high expectations for rapid growth poses a significant challenge. The company's recent Series B funding and hiring of a growth marketer may create pressure to deliver results quickly, which could be difficult for Sircles to meet. I recommend nurturing the lead to better understand their expectations and growth goals, and to assess whether Sircles can deliver results that meet their needs. The key evidence of NileBooks' interest in scaling paid acquisition and their strong growth signals supports the potential for a successful partnership, but caution is needed to ensure that Sircles can deliver results that meet NileBooks' expectations.
  - Evidence: NileBooks' interest in scaling paid acquisition; Recent Series B funding; 40% quarter-over-quarter increase in website traffic
  - Risks: High expectations for rapid growth; Potential difficulty in delivering results quickly

## 3. Resolution
- Method: `confidence_weighted_vote`
- **Disposition: `pursue` -> action: `auto_send`** (confidence 0.9)
- Winning position: Growth Advocate
- Rationale: Growth Advocate prevailed (0.90 vs 0.70 confidence) recommending 'pursue'.
- **Recorded dissent:** {
  "agent": "Risk Skeptic",
  "recommendation": "nurture",
  "confidence": 0.7,
  "rationale": "While NileBooks' strong growth signals and high ICP score are promising, the risk of high expectations for rapid growth poses a significant challenge. The company's recent Series B funding and hiring of a growth marketer may create pressure to deliver results quickly, which could be difficult for Sircles to meet. I recommend nurturing the lead to better understand their expectations and growth goals, and to assess whether Sircles can deliver results that meet their needs. The key evidence of NileBooks' interest in scaling paid acquisition and their strong growth signals supports the potential for a successful partnership, but caution is needed to ensure that Sircles can deliver results that meet NileBooks' expectations.",
  "why_it_lost": "Lower confidence (0.70) than the winner."
}

## 4. Outreach artifacts
**Subject:** Scaling Paid Acquisition at NileBooks

**Email:**

Hi Mariam, I saw that you're looking to scale paid acquisition at NileBooks. I'd love to explore how Sircles can help. Can we schedule a call to discuss?

**LinkedIn note:** Hi Mariam, excited to connect and discuss how Sircles can support NileBooks' growth goals

## 5. CRM record (mock HubSpot)
```json
{
  "lead_id": "L-001",
  "contact_name": "Mariam Hassan",
  "company": "NileBooks",
  "deal_stage": "Outreach Sent",
  "disposition": "pursue",
  "icp_score": 100,
  "owner": "Auto-pipeline",
  "flags": [],
  "next_step": "First-touch sent; monitor for reply.",
  "notes": "Internal Note: Decision to pursue NileBooks made with 0.9 confidence by Growth Advocate, outweighing Risk Skeptic's recommendation to nurture with 0.7 confidence, due to lower confidence in the dissenting opinion. The winning argument was based on NileBooks' strong growth signals and high ICP score, while the losing argument cautioned against high expectations for rapid growth following their recent Series B funding."
}
```

In [9]:
# --- Ambiguous case: L-003 (competing agency -> debate -> escalate) ---
state_ambiguous = run_pipeline(LEADS["L-003"], agents)
display(Markdown(render_trace(state_ambiguous)))


# Run trace - L-003: Omar Farouk @ BrightWave Digital
_Source: Contact form. Inbound note: Asked about our process and pricing in detail._

## 1. Enrichment & ICP score
- Contact: Head of Growth, seniority `director`, Cairo, Egypt, email verified: True
- Company: Marketing & Advertising Agency, 45 employees, Bootstrapped, agency: True
- Signals: Active on LinkedIn, Publishes client case studies
- **ICP score: 63/100** (band `nurture`) | breakdown: {'seniority': 15, 'size_budget': 20, 'industry': 0, 'location': 15, 'signals': 10, 'maturity_gap': 3}
- Disqualifiers: ['competing_agency'] | data completeness: 1.0

## 2. Debate (round 1)
- **Growth Advocate** -> `nurture` @ confidence **0.70**
  - Rationale: While there is a risk of conflict of interest due to BrightWave Digital being a competing agency, the company's suitable seniority level, size, and budget, as well as Omar Farouk's interest in Sircles' services, make it worth exploring the opportunity further
  - Evidence: Omar Farouk is the Head of Growth at BrightWave Digital; The company has a moderate ICP score of 63; BrightWave Digital has a suitable seniority level, size, and budget; The company is active on LinkedIn and publishes client case studies
  - Risks: Conflict of interest due to competing agency
- **Risk Skeptic** -> `skip` @ confidence **0.80**
  - Rationale: While Omar Farouk and BrightWave Digital show some promising signs, the fact that they are a competing agency poses a significant conflict of interest risk. Their interest in Sircles' services may be for competitive intelligence rather than a genuine need, which could lead to a waste of resources and potential reputational damage.
  - Evidence: Moderate ICP score of 63; Suitable seniority level, size, and budget; Active engagement on LinkedIn; Competing agency
  - Risks: Conflict of interest; Waste of resources; Reputational damage

## 3. Resolution
- Method: `critic_veto:competing_agency`
- **Disposition: `skip` -> action: `human_review`** (confidence 0.9)
- Winning position: Risk Skeptic
- Rationale: Lead is itself a marketing agency. Could be a competitor or a partnership/white-label opportunity - not an autopilot decision. Escalating to a human.
- **Recorded dissent:** {
  "agent": "Growth Advocate",
  "recommendation": "nurture",
  "confidence": 0.7,
  "rationale": "While there is a risk of conflict of interest due to BrightWave Digital being a competing agency, the company's suitable seniority level, size, and budget, as well as Omar Farouk's interest in Sircles' services, make it worth exploring the opportunity further",
  "why_it_lost": "Veto on competitor conflict overrides the growth case."
}
- **ESCALATED TO HUMAN:** Competing agency - potential conflict or partnership

## 4. Outreach artifacts
_None - no autonomous first-touch for this disposition._

## 5. CRM record (mock HubSpot)
```json
{
  "lead_id": "L-003",
  "contact_name": "Omar Farouk",
  "company": "BrightWave Digital",
  "deal_stage": "Needs Human Review",
  "disposition": "skip",
  "icp_score": 63,
  "owner": "Human (SDR lead)",
  "flags": [
    "competing_agency",
    "escalate:Competing agency - potential conflict or partnership"
  ],
  "next_step": "Route to a human for a judgement call.",
  "notes": "Internal Note: Decision to skip lead Omar Farouk from BrightWave Digital due to competing agency concerns, overriding Growth Advocate's recommendation to nurture based on suitable seniority, size, and budget. The Risk Skeptic's veto on competitor conflict took precedence, citing potential conflict or partnership, and the lead has been escalated for human review."
}
```

In [10]:
# --- Summary across all four sample leads ---
import pandas as pd  # optional; falls back to plain print if unavailable

rows = []
for lead in fetch_new_leads():
    s = run_pipeline(lead, agents)
    rows.append({
        "lead": lead.lead_id,
        "company": s.enrichment.company.name,
        "ICP": s.icp.total,
        "band": s.icp.band,
        "disposition": s.resolution.disposition,
        "action": s.resolution.action,
        "conf": s.resolution.confidence,
        "method": s.resolution.method,
        "escalated": s.resolution.escalate,
    })

try:
    display(pd.DataFrame(rows))
except Exception:
    for r in rows:
        print(r)


,lead,company,ICP,band,disposition,action,conf,method,escalated
0,L-001,NileBooks,100,pursue,pursue,auto_send,0.90,confidence_weighted_vote,False
1,L-002,Self-employed,13,skip,skip,archive,0.90,hard_disqualifier:no_budget_micro,False
2,L-003,BrightWave Digital,63,nurture,skip,human_review,0.90,critic_veto:competing_agency,True
3,L-004,Zeitoun,74,nurture,nurture,nurture_sequence,0.85,consensus,False


## 9. Tests

The two highest-value things to cover (per the brief) are the **resolution rule** and **one agent's contract**. Both are deterministic, so these tests run **without an API key**.

- **Resolution rule** - competitor veto, hard disqualifier, consensus, confidence-weighted vote, contested escalation, low-data escalation, and the confidence->action gating thresholds.
- **Lead Intelligence contract** - `score_icp` returns the expected band/disqualifiers for each sample lead, and the `Position` schema rejects invalid confidence.

The cell below is a self-contained harness: it discovers every `test_*` function and reports pass/fail, and `assert not _failures` makes the cell fail loudly in CI or "Run All".

In [11]:
from sircles_core import _gate_action  # private helper exercised by tests
from pydantic import ValidationError


def mk_icp(disq=None, band="pursue", data=1.0) -> ICPScore:
    return ICPScore(total=80, breakdown={}, band=band, disqualifiers=disq or [], data_completeness=data)


def mk_pos(agent, rec, conf) -> Position:
    return Position(agent=agent, recommendation=rec, confidence=conf, rationale="test")


# ---------- Resolution-rule tests ----------
def test_competitor_veto_escalates():
    r = resolve(mk_icp(disq=["competing_agency"]),
                mk_pos("Growth Advocate", "pursue", 0.85), mk_pos("Risk Skeptic", "skip", 0.7))
    assert r.escalate and r.action == "human_review"
    assert r.method.startswith("critic_veto") and r.dissent is not None


def test_hard_disqualifier_skips():
    r = resolve(mk_icp(disq=["no_budget_micro"], band="skip"),
                mk_pos("Growth Advocate", "nurture", 0.6), mk_pos("Risk Skeptic", "skip", 0.8))
    assert r.disposition == "skip" and r.action == "archive" and not r.escalate


def test_consensus_high_conf_auto_sends():
    r = resolve(mk_icp(), mk_pos("A", "pursue", 0.8), mk_pos("S", "pursue", 0.7))
    assert r.method == "consensus" and r.disposition == "pursue" and r.action == "auto_send"


def test_consensus_low_conf_needs_review():
    r = resolve(mk_icp(), mk_pos("A", "pursue", 0.5), mk_pos("S", "pursue", 0.5))
    assert r.disposition == "pursue" and r.action == "draft_for_review"


def test_confidence_weighted_vote_picks_winner():
    r = resolve(mk_icp(), mk_pos("Growth Advocate", "pursue", 0.9), mk_pos("Risk Skeptic", "skip", 0.6))
    assert r.method == "confidence_weighted_vote" and r.disposition == "pursue"
    assert r.winning_agent == "Growth Advocate" and r.dissent["agent"] == "Risk Skeptic"


def test_contested_polar_triggers_rebuttal_then_escalates():
    # Polar split (pursue vs skip) with comparable confidence -> rebuttal -> escalate.
    adv, skep = mk_pos("Growth Advocate", "pursue", 0.70), mk_pos("Risk Skeptic", "skip", 0.62)
    first = resolve(mk_icp(), adv, skep, rebuttal_done=False)
    assert first.method == "contested:needs_rebuttal" and not first.escalate
    final = resolve(mk_icp(), adv, skep, rebuttal_done=True)
    assert final.escalate and final.method == "contested_escalation" and final.action == "human_review"


def test_adjacent_split_uses_icp_prior():
    # Adjacent split (pursue vs nurture) with close confidence -> ICP band breaks the tie, no escalation.
    icp = mk_icp(band="pursue")
    r = resolve(icp, mk_pos("Growth Advocate", "pursue", 0.80),
                mk_pos("Risk Skeptic", "nurture", 0.70), rebuttal_done=True)
    assert r.method == "icp_prior_tiebreak" and r.disposition == "pursue"
    assert r.action == "auto_send" and not r.escalate and r.winning_agent == "Growth Advocate"


def test_low_data_escalates():
    r = resolve(mk_icp(data=0.3), mk_pos("A", "pursue", 0.8), mk_pos("S", "skip", 0.8))
    assert r.escalate and r.method == "low_data_confidence"


def test_action_gating_thresholds():
    assert _gate_action("pursue", 0.70, False) == "auto_send"
    assert _gate_action("pursue", 0.69, False) == "draft_for_review"
    assert _gate_action("nurture", 0.99, False) == "nurture_sequence"
    assert _gate_action("skip", 0.99, False) == "archive"
    assert _gate_action("pursue", 0.99, True) == "human_review"


# ---------- Lead Intelligence (ICP) contract tests ----------
def _icp_for(lead_id: str) -> ICPScore:
    lead = {l.lead_id: l for l in fetch_new_leads()}[lead_id]
    return score_icp(Enrichment(contact=enrich_contact(lead), company=enrich_company(lead)))


def test_icp_bands_and_disqualifiers():
    c = _icp_for("L-001"); assert c.band == "pursue" and not c.disqualifiers and 0 <= c.total <= 100
    s = _icp_for("L-002"); assert s.band == "skip" and "no_budget_micro" in s.disqualifiers
    a = _icp_for("L-003"); assert "competing_agency" in a.disqualifiers
    n = _icp_for("L-004"); assert n.band == "nurture"  # demoted by the no-budget rule


def test_position_rejects_bad_confidence():
    try:
        Position(recommendation="pursue", confidence=1.5, rationale="x")
        assert False, "expected ValidationError for confidence > 1"
    except ValidationError:
        pass


# ---------- Runner ----------
_TESTS = [v for k, v in sorted(globals().items()) if k.startswith("test_") and callable(v)]
_failures = []
for t in _TESTS:
    try:
        t()
        print(f"PASS  {t.__name__}")
    except AssertionError as e:
        _failures.append((t.__name__, str(e)))
        print(f"FAIL  {t.__name__}: {e}")

print("\n" + ("All tests passed." if not _failures else f"{len(_failures)} test(s) failed."))
assert not _failures

PASS  test_action_gating_thresholds
PASS  test_adjacent_split_uses_icp_prior
PASS  test_competitor_veto_escalates
PASS  test_confidence_weighted_vote_picks_winner
PASS  test_consensus_high_conf_auto_sends
PASS  test_consensus_low_conf_needs_review
PASS  test_contested_polar_triggers_rebuttal_then_escalates
PASS  test_hard_disqualifier_skips
PASS  test_icp_bands_and_disqualifiers
PASS  test_low_data_escalates
PASS  test_position_rejects_bad_confidence

All tests passed.
